## SRP064240

**paper:** [PMID: 27703277](https://pmc.ncbi.nlm.nih.gov/articles/PMC5050510/) - Repeatability of cortisol stress response in the European sea bass (Dicentrarchus labrax) and transcription differences between individuals with divergent responses, 2016

**date, curator:** 2026-06-03, Sara Carsanaro

**notes**
* the stress test was chasing the fish with a net for 5 minutes, find to add as healthy wild type
* dev stage: immature fish (17 months-old at the beginning of the experiment), annotating as juvenile
* protocol is TruSeq v2 RNA, mRNA seq but no additional details, just annotating as full_length and polyA

### annotation summary

In [23]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Liver,UBERON:0002107,liver,perfect match


In [24]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,642 dph,UBERON:0034919,juvenile stage,missing child term
3,631 dph,UBERON:0034919,juvenile stage,missing child term


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP064240"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|█████████████████████████████████████████████| 6/6 [00:06<00:00,  1.03s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [26]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1292227,SRP064240,Illumina HiSeq 2000,SRS1091672,,,,,,Liver,642 dph,,,,M,,,13489,,,,,,Liv_RNA-59,SAMN04110149,642,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-3,,,,not applicable,,TRANSCRIPTOMIC,unspecified
1,SRX1292226,SRP064240,Illumina HiSeq 2000,SRS1091673,,,,,,Liver,642 dph,,,,M,,,13489,,,,,,Liv_RNA-52,SAMN04110148,642,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-2,,,,not applicable,,TRANSCRIPTOMIC,unspecified
2,SRX1292225,SRP064240,Illumina HiSeq 2000,SRS1091674,,,,,,Liver,642 dph,,,,M,,,13489,,,,,,Liv_RNA-50,SAMN04110147,642,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-1,,,,not applicable,,TRANSCRIPTOMIC,unspecified
3,SRX1292224,SRP064240,Illumina HiSeq 2000,SRS1091675,,,,,,Liver,631 dph,,,,M,,,13489,,,,,,Liv_RNA-39,SAMN04110146,631,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_HR-3,,,,not applicable,,TRANSCRIPTOMIC,unspecified
4,SRX1292223,SRP064240,Illumina HiSeq 2000,SRS1091676,,,,,,Liver,631 dph,,,,F,,,13489,,,,,,Liv_RNA-20,SAMN04110145,631,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_HR-2,,,,not applicable,,TRANSCRIPTOMIC,unspecified
5,SRX1292222,SRP064240,Illumina HiSeq 2000,SRS1091677,,,,,,Liver,631 dph,,,,M,,,13489,,,,,,Liv_RNA-18,SAMN04110144,631,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_HR-1,,,,not applicable,,TRANSCRIPTOMIC,unspecified


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [27]:
unique_sorted(library, "infoOrgan")

['Liver']


In [28]:

# all
library.loc[:,'anatId'] = 'UBERON:0002107'
library.loc[:,'anatName'] = 'liver'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1292227,SRP064240,Illumina HiSeq 2000,SRS1091672,UBERON:0002107,liver,,,,Liver,642 dph,perfect match,not documented,,M,,,13489,,,,,,Liv_RNA-59,SAMN04110149,642,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-3,,,,not applicable,,TRANSCRIPTOMIC,unspecified
1,SRX1292226,SRP064240,Illumina HiSeq 2000,SRS1091673,UBERON:0002107,liver,,,,Liver,642 dph,perfect match,not documented,,M,,,13489,,,,,,Liv_RNA-52,SAMN04110148,642,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-2,,,,not applicable,,TRANSCRIPTOMIC,unspecified
2,SRX1292225,SRP064240,Illumina HiSeq 2000,SRS1091674,UBERON:0002107,liver,,,,Liver,642 dph,perfect match,not documented,,M,,,13489,,,,,,Liv_RNA-50,SAMN04110147,642,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-1,,,,not applicable,,TRANSCRIPTOMIC,unspecified
3,SRX1292224,SRP064240,Illumina HiSeq 2000,SRS1091675,UBERON:0002107,liver,,,,Liver,631 dph,perfect match,not documented,,M,,,13489,,,,,,Liv_RNA-39,SAMN04110146,631,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_HR-3,,,,not applicable,,TRANSCRIPTOMIC,unspecified
4,SRX1292223,SRP064240,Illumina HiSeq 2000,SRS1091676,UBERON:0002107,liver,,,,Liver,631 dph,perfect match,not documented,,F,,,13489,,,,,,Liv_RNA-20,SAMN04110145,631,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_HR-2,,,,not applicable,,TRANSCRIPTOMIC,unspecified
5,SRX1292222,SRP064240,Illumina HiSeq 2000,SRS1091677,UBERON:0002107,liver,,,,Liver,631 dph,perfect match,not documented,,M,,,13489,,,,,,Liv_RNA-18,SAMN04110144,631,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_HR-1,,,,not applicable,,TRANSCRIPTOMIC,unspecified


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [29]:
unique_sorted(library, "infoStage")

['631 dph' '642 dph']


In [30]:
# all
library.loc[:,'stageId'] = 'UBERON:0034919'
library.loc[:,'stageName'] = 'juvenile stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'missing child term'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1292227,SRP064240,Illumina HiSeq 2000,SRS1091672,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,,,,,Liv_RNA-59,SAMN04110149,642,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-3,,,,not applicable,,TRANSCRIPTOMIC,unspecified
1,SRX1292226,SRP064240,Illumina HiSeq 2000,SRS1091673,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,,,,,Liv_RNA-52,SAMN04110148,642,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-2,,,,not applicable,,TRANSCRIPTOMIC,unspecified
2,SRX1292225,SRP064240,Illumina HiSeq 2000,SRS1091674,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,,,,,Liv_RNA-50,SAMN04110147,642,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-1,,,,not applicable,,TRANSCRIPTOMIC,unspecified
3,SRX1292224,SRP064240,Illumina HiSeq 2000,SRS1091675,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,631 dph,perfect match,not documented,missing child term,M,,,13489,,,,,,Liv_RNA-39,SAMN04110146,631,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_HR-3,,,,not applicable,,TRANSCRIPTOMIC,unspecified
4,SRX1292223,SRP064240,Illumina HiSeq 2000,SRS1091676,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,631 dph,perfect match,not documented,missing child term,F,,,13489,,,,,,Liv_RNA-20,SAMN04110145,631,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_HR-2,,,,not applicable,,TRANSCRIPTOMIC,unspecified
5,SRX1292222,SRP064240,Illumina HiSeq 2000,SRS1091677,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,631 dph,perfect match,not documented,missing child term,M,,,13489,,,,,,Liv_RNA-18,SAMN04110144,631,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Index

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [31]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
my_protocolType = 'full_length'

#library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1292227,SRP064240,Illumina HiSeq 2000,SRS1091672,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-59,SAMN04110149,642,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-3,,,,not applicable,,TRANSCRIPTOMIC,unspecified
1,SRX1292226,SRP064240,Illumina HiSeq 2000,SRS1091673,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-52,SAMN04110148,642,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-2,,,,not applicable,,TRANSCRIPTOMIC,unspecified
2,SRX1292225,SRP064240,Illumina HiSeq 2000,SRS1091674,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-50,SAMN04110147,642,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-1,,,,not applicable,,TRANSCRIPTOMIC,unspecified
3,SRX1292224,SRP064240,Illumina HiSeq 2000,SRS1091675,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,631 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-39,SAMN04110146,631,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_HR-3,,,,not applicable,,TRANSCRIPTOMIC,unspecified
4,SRX1292223,SRP064240,Illumina HiSeq 2000,SRS1091676,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,631 dph,perfect match,not documented,missing child term,F,,,13489,,full_length,polyA,,,Liv_RNA-20,SAMN04110145,631,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_HR-2,,,,not applicable,,TRANSCRIPTOMIC,unspecified
5,SRX1292222,SRP064240,Illumina HiSeq 2000,SRS1091677,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,631 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-18,SAMN04110144,631,days post hatch,,,,,,stressed - chased with net for 5 min,,,03/06/2026,"""Sequencing libraries were prepared from mRN

#### globin, replicates

In [32]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [33]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-06-03'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1292227,SRP064240,Illumina HiSeq 2000,SRS1091672,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-59,SAMN04110149,642,days post hatch,,,,,,stressed - chased with net for 5 min,,SAC,2026-06-03,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-3,,,,not applicable,,TRANSCRIPTOMIC,unspecified
1,SRX1292226,SRP064240,Illumina HiSeq 2000,SRS1091673,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-52,SAMN04110148,642,days post hatch,,,,,,stressed - chased with net for 5 min,,SAC,2026-06-03,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-2,,,,not applicable,,TRANSCRIPTOMIC,unspecified
2,SRX1292225,SRP064240,Illumina HiSeq 2000,SRS1091674,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-50,SAMN04110147,642,days post hatch,,,,,,stressed - chased with net for 5 min,,SAC,2026-06-03,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_LR-1,,,,not applicable,,TRANSCRIPTOMIC,unspecified
3,SRX1292224,SRP064240,Illumina HiSeq 2000,SRS1091675,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,631 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-39,SAMN04110146,631,days post hatch,,,,,,stressed - chased with net for 5 min,,SAC,2026-06-03,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_HR-3,,,,not applicable,,TRANSCRIPTOMIC,unspecified
4,SRX1292223,SRP064240,Illumina HiSeq 2000,SRS1091676,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,631 dph,perfect match,not documented,missing child term,F,,,13489,,full_length,polyA,,,Liv_RNA-20,SAMN04110145,631,days post hatch,,,,,,stressed - chased with net for 5 min,,SAC,2026-06-03,"""Sequencing libraries were prepared from mRNA using TruSeq v2 RNA reagents (Illumina), with 4 minutes fragmentation and 15 cycles PCR. Indexed libraries were sequenced over a total of five lanes on a HiSeq 2000 (Illumina), employing 125 bp paired end reads. """,,DL_Liv_HR-2,,,,not applicable,,TRANSCRIPTOMIC,unspecified
5,SRX1292222,SRP064240,Illumina HiSeq 2000,SRS1091677,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,631 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-18,SAMN04110144,631,days post hatch,,,,,,stressed - chased with net for 5 min,,SAC,2026-06-03,"""Sequencing libraries were

#### comments

In [34]:
library.loc[:,'comment'] = 'PMID: 27703277'

#### save complete file with correct columns

In [35]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX1292227,SRP064240,Illumina HiSeq 2000,SRS1091672,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-59,SAMN04110149,642,days post hatch,,,,,PMID: 27703277,stressed - chased with net for 5 min,,SAC,2026-06-03
1,SRX1292226,SRP064240,Illumina HiSeq 2000,SRS1091673,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-52,SAMN04110148,642,days post hatch,,,,,PMID: 27703277,stressed - chased with net for 5 min,,SAC,2026-06-03
2,SRX1292225,SRP064240,Illumina HiSeq 2000,SRS1091674,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-50,SAMN04110147,642,days post hatch,,,,,PMID: 27703277,stressed - chased with net for 5 min,,SAC,2026-06-03
3,SRX1292224,SRP064240,Illumina HiSeq 2000,SRS1091675,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,631 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-39,SAMN04110146,631,days post hatch,,,,,PMID: 27703277,stressed - chased with net for 5 min,,SAC,2026-06-03
4,SRX1292223,SRP064240,Illumina HiSeq 2000,SRS1091676,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,631 dph,perfect match,not documented,missing child term,F,,,13489,,full_length,polyA,,,Liv_RNA-20,SAMN04110145,631,days post hatch,,,,,PMID: 27703277,stressed - chased with net for 5 min,,SAC,2026-06-03
5,SRX1292222,SRP064240,Illumina HiSeq 2000,SRS1091677,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,631 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-18,SAMN04110144,631,days post hatch,,,,,PMID: 27703277,stressed - chased with net for 5 min,,SAC,2026-06-03


### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP064240,Dicentrarchus labrax Transcriptome or Gene expression,Transcriptome profile of high and low stress responders in the European sea bass.,SRA,,,,,,,PRJNA296857,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

6

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP064240,Dicentrarchus labrax Transcriptome or Gene expression,Transcriptome profile of high and low stress responders in the European sea bass.,SRA,total,Bgee 1K,6,,full_length,,PRJNA296857,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '27703277'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC5050510/'
experiment.loc[:,'DOI'] = '10.1038/srep34858'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP064240,Dicentrarchus labrax Transcriptome or Gene expression,Transcriptome profile of high and low stress responders in the European sea bass.,SRA,total,Bgee 1K,6,,full_length,,PRJNA296857,27703277,https://pmc.ncbi.nlm.nih.gov/articles/PMC5050510/,10.1038/srep34858,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [36]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [37]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 6
Errors: 6
Warnings: 0
Top codes:
  - BAD_VALUE: 6


#### check columns match

In [38]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [39]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
65100,ERX8655526,ERP134561,Illumina NovaSeq 6000,ERS9451783,UBERON:0006728,liver left medial lobe,UBERON:0000113,post-juvenile adult stage,,Liver (left medial lobe),Adult,perfect match,not documented,perfect match,F,,,10090,TruSeq Stranded mRNA,full_length,polyA,,,DOM21Liv,SAMEA11804451,,,,,,,"PMID:40960491, Nine age-matched adult females ...",,,ANN,2026-06-01
65101,ERX8656406,ERP134561,Illumina NovaSeq 6000,ERS9452663,UBERON:0009022,right uterine horn,UBERON:0000113,post-juvenile adult stage,,Uterus (right uterine horn),Adult,perfect match,not documented,perfect match,F,,,10103,TruSeq Stranded mRNA,full_length,polyA,,,SPI23Ute,SAMEA11805332,,,,,,,"PMID:40960491, Nine age-matched adult females ...",,,ANN,2026-06-01
65102,SRX1292227,SRP064240,Illumina HiSeq 2000,SRS1091672,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-59,SAMN04110149,642,days post hatch,,,,,PMID: 27703277,stressed - chased with net for 5 min,,SAC,2026-06-03
65103,SRX1292226,SRP064240,Illumina HiSeq 2000,SRS1091673,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-52,SAMN04110148,642,days post hatch,,,,,PMID: 27703277,stressed - chased with net for 5 min,,SAC,2026-06-03
65104,SRX1292225,SRP064240,Illumina HiSeq 2000,SRS1091674,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,642 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-50,SAMN04110147,642,days post hatch,,,,,PMID: 27703277,stressed - chased with net for 5 min,,SAC,2026-06-03
65105,SRX1292224,SRP064240,Illumina HiSeq 2000,SRS1091675,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,631 dph,perfect match,not documented,missing child term,M,,,13489,,full_length,polyA,,,Liv_RNA-39,SAMN04110146,631,days post hatch,,,,,PMID: 27703277,stressed - chased with net for 5 min,,SAC,2026-06-03
65106,SRX1292223,SRP064240,Illumina HiSeq 2000,SRS1091676,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,Liver,631 dph,perfect match,not documented,missing child term,F,,,13489,,full_length,polyA,,,Liv_RNA-20,SAMN04110145,631,days post hatch,,,,,PMID: 27703277,stressed - chased with net for 5 min,,SAC,2026-06-03


In [40]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1240,SRP511887,"Landscape genomics, climate change and adaptation",The central aim of the project is to determine...,SRA,partial,Bgee 1K,139,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA1120248,27599812,https://pmc.ncbi.nlm.nih.gov/articles/PMC5013434/,10.1038/srep32975,,[Bgee curator notes: PMID:27599812 has link to...
1241,ERP134561,RNA-Seq of samples from four wild mouse taxa,Illumina RNA-Seq data were collected from outb...,SRA,total,Bgee 1K,1364,TruSeq Stranded mRNA,full_length,,PRJEB50011,40960491,https://pmc.ncbi.nlm.nih.gov/articles/PMC12443...,"10.1101/2024.05.22.595301,10.7554/eLife.99602....",40926263[PMID],
1242,SRP064240,Dicentrarchus labrax Transcriptome or Gene exp...,Transcriptome profile of high and low stress r...,SRA,total,Bgee 1K,6,,full_length,,PRJNA296857,27703277,https://pmc.ncbi.nlm.nih.gov/articles/PMC5050510/,10.1038/srep34858,,


### add annotations to git

In [41]:
! git pull

Already up to date.


In [42]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [43]:
! git add $git_experiment_path $git_library_path

In [44]:
! git commit -m $commit_message_exp

[develop 3d22955] adding annotated bulk experiment SRP064240
 2 files changed, 7 insertions(+)


In [45]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.09 KiB | 2.09 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   a6c18e1..3d22955  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push